# FastAPI SSE Streaming — Token-by-Token LangGraph Output
## Stream Tokens Over HTTP — FastAPI and Server-Sent Events
⏱ ~15 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/75-fastapi-sse-streaming/fastapi_sse_streaming_workbook.ipynb)

Streams LangGraph output token-by-token using astream_events and Server-Sent Events (SSE). The notebook demos the streaming graph in Colab; the FastAPI server component is shown as reference code.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — SSE vs WebSocket; streaming=True; astream_events protocol |
| 2 | **Streaming Graph** — think → answer nodes; streaming=True in ChatOpenAI |
| 3 | **astream_events()** — on_chat_model_stream events for token-level output |
| 4 | **FastAPI Reference** — EventSourceResponse pattern (reference — not run in Colab) |
| 5 | **Full Demo** — Live token streaming in Colab cell |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

## Part 1 — Streaming Fundamentals

### Why stream at all?

Without streaming, users wait for the entire response before seeing anything — even if the LLM starts generating immediately. For a 500-word answer, that's a 10-second blank screen.

Streaming delivers tokens as they arrive. The perceived latency drops to the time-to-first-token (TTFT), which is typically under 1 second.

### SSE vs WebSocket vs Polling

| Method | Direction | Use case |
|--------|-----------|----------|
| **Server-Sent Events (SSE)** | Server → Client, one-way | LLM token streams, live feeds |
| **WebSocket** | Bidirectional | Chat, multiplayer, collaborative editing |
| **Polling** | Client pulls on interval | Simple status checks, legacy APIs |

SSE is the right choice for LLM output: it's one-way (server pushes tokens), works over plain HTTP (no upgrade needed), and auto-reconnects on connection drop.

### The `astream_events` protocol

LangGraph's `astream_events()` yields typed event dicts. The key events:

| Event name | When it fires |
|-----------|--------------|
| `on_chain_start` | Graph invocation begins |
| `on_node_start` | A graph node starts executing |
| `on_chat_model_stream` | One token arrives from the LLM |
| `on_tool_start` | A tool call begins |
| `on_chain_end` | Graph invocation completes |

You typically filter on `on_chat_model_stream` to print tokens, and optionally `on_node_start` to show progress indicators.

> **Prerequisite:** You've built LangGraph pipelines and async Python before (see examples 62, 81).

In [ ]:
import asyncio
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

# --- minimal streaming demo: one node, one LLM call ---
class SimpleState(TypedDict):
    question: str
    answer:   str

llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)

def answer_node(state: SimpleState) -> dict:
    r = llm.invoke([HumanMessage(content=state["question"])])
    return {"answer": r.content.strip()}

g = StateGraph(SimpleState)
g.add_node("answer", answer_node)
g.add_edge(START, "answer")
g.add_edge("answer", END)
simple_app = g.compile()

# --- stream and print all events ---
async def show_all_events(question: str):
    print(f"Events for: {question!r}\n")
    async for event in simple_app.astream_events(
        {"question": question, "answer": ""}, version="v1"
    ):
        name = event["event"]
        # skip noisy internal events — show only the meaningful ones
        if name in ("on_chain_start", "on_node_start", "on_chat_model_stream", "on_chain_end"):
            data_preview = ""
            if name == "on_chat_model_stream":
                data_preview = repr(event["data"]["chunk"].content[:30])
            print(f"  {name:<30} {data_preview}")

await show_all_events("What is Python in one sentence?")

## Part 2 — Token-by-Token Streaming

Filtering `on_chat_model_stream` events gives you each token as it arrives. This is what drives the "typing" effect you see in ChatGPT and Claude.

Key details:
- `event["data"]["chunk"].content` is the token string (may be empty for metadata chunks — always guard with `if chunk`)
- Tokens arrive in order — concatenating them reconstructs the full response
- The graph runs the node synchronously inside; streaming is handled at the `astream_events` layer
- `version="v1"` is the stable event schema — `v2` exists but changes event field names

### Multi-node streaming

When your graph has multiple LLM nodes, `on_chat_model_stream` fires for tokens from *all* of them. Use `event["metadata"]["langgraph_node"]` to identify which node the token came from.

In [ ]:
import asyncio, time
from typing import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

class ThinkAnswerState(TypedDict):
    question: str
    thinking: str
    answer:   str

llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)

def think_node(state: ThinkAnswerState) -> dict:
    r = llm.invoke([
        SystemMessage(content="Think briefly about the question in 1-2 sentences."),
        HumanMessage(content=state["question"]),
    ])
    return {"thinking": r.content.strip()}

def answer_node(state: ThinkAnswerState) -> dict:
    r = llm.invoke([
        SystemMessage(content="Give a concise, direct answer."),
        HumanMessage(content=state["question"]),
    ])
    return {"answer": r.content.strip()}

g2 = StateGraph(ThinkAnswerState)
g2.add_node("think_node",  think_node)
g2.add_node("answer_node", answer_node)
g2.add_edge(START, "think_node")
g2.add_edge("think_node", "answer_node")
g2.add_edge("answer_node", END)
app2 = g2.compile()

DEMO_QUESTION = "Explain how photosynthesis works in 2 sentences."

async def stream_with_node_labels(question: str):
    t0           = time.time()
    token_count  = 0
    current_node = None

    async for event in app2.astream_events(
        {"question": question, "thinking": "", "answer": ""},
        version="v1",
    ):
        ename = event["event"]

        if ename == "on_node_start":
            node = event["name"]
            if node != current_node:
                if current_node:
                    print()   # newline after previous node's tokens
                print(f"\n[{node}] ", end="", flush=True)
                current_node = node

        elif ename == "on_chat_model_stream":
            chunk = event["data"]["chunk"].content
            if chunk:
                print(chunk, end="", flush=True)
                token_count += 1

    elapsed = time.time() - t0
    print(f"\n\n({elapsed:.1f}s, {token_count} tokens from {2} LLM nodes)")

await stream_with_node_labels(DEMO_QUESTION)

## Part 3 — Batch vs Stream Latency Comparison

The difference between `invoke()` and `astream_events()` is not about what the model does — it's about *when you see the output*.

- `invoke()` blocks until the full response is ready, then returns the complete string
- `astream_events()` yields tokens as they arrive from the API

For a 200-token response at 50 tokens/sec:
- **Batch:** user waits ~4 seconds, then sees everything at once
- **Stream:** user sees the first token in ~0.2s, full response in ~4s — same total time, much better perceived speed

### When NOT to stream

- Structured output that must be fully parsed before use (e.g. JSON tool calls)
- Short responses under ~20 tokens — the overhead isn't worth it
- Batch processing pipelines where no human is watching

In [ ]:
import asyncio, time
from langchain_core.messages import HumanMessage

QUESTION = "List 5 benefits of using a vector database for RAG."

# --- batch invoke ---
t0     = time.time()
result = app2.invoke({"question": QUESTION, "thinking": "", "answer": ""})
batch_time = time.time() - t0
print(f"Batch invoke:    {batch_time:.2f}s to first character")
print(f"Answer preview:  {result['answer'][:100]}...\n")

# --- streaming ---
async def time_to_first_token(question: str):
    t0   = time.time()
    ttft = None
    all_chunks = []
    async for event in app2.astream_events(
        {"question": question, "thinking": "", "answer": ""},
        version="v1",
    ):
        if event["event"] == "on_chat_model_stream":
            chunk = event["data"]["chunk"].content
            if chunk:
                if ttft is None:
                    ttft = time.time() - t0
                all_chunks.append(chunk)
    total = time.time() - t0
    return ttft, total, "".join(all_chunks)

ttft, total, full = await time_to_first_token(QUESTION)
print(f"Stream TTFT:     {ttft:.2f}s to first token")
print(f"Stream total:    {total:.2f}s to last token")
print(f"Answer preview:  {full[:100]}...")
print(f"\nPerceived speedup: {batch_time / ttft:.1f}x faster first-character appearance")

## Part 4 — FastAPI SSE Server Pattern

The streaming graph from Part 2 can be exposed over HTTP using FastAPI and `sse-starlette`. Each HTTP request triggers a `astream_events()` call, and each token becomes a Server-Sent Event frame.

The client (browser or curl) receives a `text/event-stream` response and processes events as they arrive — no polling, no WebSocket upgrade.

```
Client                        FastAPI Server                    OpenAI API
  |--- GET /stream?q=... ---->|                                    |
  |                           |--- astream_events() ----------->   |
  |<-- data: token1 ----------|<-- chunk ---------------------     |
  |<-- data: token2 ----------|<-- chunk ---------------------     |
  |<-- data: [DONE] ----------|                                    |
```

> **This cell is reference code only** — run it locally with `uvicorn`, not in Colab (Colab can't bind a public port without ngrok).

In [ ]:
# FastAPI SSE server — reference implementation (run locally, not in Colab)
FASTAPI_REFERENCE = '''
import asyncio
from fastapi import FastAPI
from sse_starlette.sse import EventSourceResponse
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from typing import TypedDict

# --- build the same graph as Part 2 ---
class ThinkAnswerState(TypedDict):
    question: str; thinking: str; answer: str

llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)

def think_node(state): ...   # same as Part 2
def answer_node(state): ...  # same as Part 2

g = StateGraph(ThinkAnswerState)
# ... (same graph wiring as Part 2)
app = g.compile()

api = FastAPI()

@api.get("/stream")
async def stream_endpoint(question: str):
    async def generate():
        node_label = ""
        async for event in app.astream_events(
            {"question": question, "thinking": "", "answer": ""},
            version="v1",
        ):
            if event["event"] == "on_node_start":
                node_label = event["name"]
                yield {"event": "node", "data": node_label}
            elif event["event"] == "on_chat_model_stream":
                chunk = event["data"]["chunk"].content
                if chunk:
                    yield {"event": "token", "data": chunk}
        yield {"event": "done", "data": "[DONE]"}
    return EventSourceResponse(generate())

# Run with: pip install sse-starlette uvicorn
#           uvicorn main:api --host 0.0.0.0 --port 8000
# Test with: curl -N "http://localhost:8000/stream?question=Hello"
#   or in browser: new EventSource("/stream?question=Hello")
'''

print("FastAPI SSE server reference code:")
print(FASTAPI_REFERENCE)

## Exercises

---

**Exercise 1 — Count Tokens per Node**

Modify the streaming loop from Part 2 to count how many tokens each node (`think_node` vs `answer_node`) emits. Print the per-node count at the end. Which node is more verbose?

*Hint:* Use `event["metadata"].get("langgraph_node")` on `on_chat_model_stream` events.

---

**Exercise 2 — Collect the Full Streamed Answer**

Modify `stream_with_node_labels()` so that it returns the full answer string (not just the thinking) by collecting all tokens from `answer_node` and joining them.

---

**Exercise 3 — Add a Progress Indicator**

Add a `on_node_start` handler that prints `"[thinking...]"` when `think_node` starts and `"[answering...]"` when `answer_node` starts. This simulates a real UI progress indicator.

---

**Exercise 4 — Build the HTML Client**

Write a minimal HTML file with a `<script>` block that:
1. Creates a `new EventSource("/stream?question=What+is+LangGraph")`
2. Appends `token` events to a `<div>` on the page
3. Shows `[DONE]` when the `done` event fires

This is the frontend counterpart to the FastAPI SSE server in Part 4.

In [ ]:
# Exercise 1 — Count tokens per node
import asyncio

async def count_tokens_per_node(question: str):
    node_tokens: dict[str, int] = {}
    current_node = None

    async for event in app2.astream_events(
        {"question": question, "thinking": "", "answer": ""},
        version="v1",
    ):
        if event["event"] == "on_node_start":
            current_node = event["name"]
            if current_node not in node_tokens:
                node_tokens[current_node] = 0
        elif event["event"] == "on_chat_model_stream":
            chunk = event["data"]["chunk"].content
            if chunk and current_node:
                node_tokens[current_node] += 1

    print("Token counts per node:")
    for node, count in node_tokens.items():
        print(f"  {node}: {count} tokens")

await count_tokens_per_node("Explain recursion in two sentences.")

In [ ]:
# Exercise 4 — HTML Client (reference)
HTML_CLIENT = """<!DOCTYPE html>
<html>
<head><title>LangGraph SSE Stream</title></head>
<body>
  <h2>LangGraph Token Stream</h2>
  <div id="output" style="font-family: monospace; white-space: pre-wrap;"></div>
  <script>
    const output = document.getElementById('output');
    const es = new EventSource('/stream?question=What+is+LangGraph%3F');

    es.addEventListener('node', e => {
      output.textContent += '\n[' + e.data + ']\n';
    });

    es.addEventListener('token', e => {
      output.textContent += e.data;
    });

    es.addEventListener('done', e => {
      output.textContent += '\n\n[DONE]';
      es.close();
    });

    es.onerror = () => { output.textContent += '\n[connection error]'; };
  </script>
</body>
</html>"""

# Save to file to use alongside the FastAPI server from Part 4
with open("/tmp/sse_client.html", "w") as f:
    f.write(HTML_CLIENT)
print("Saved to /tmp/sse_client.html — open with the FastAPI server running on port 8000.")
print()
print(HTML_CLIENT)

---
## Answer Key

Attempt the exercises before reading below.

In [ ]:
# Answer 2 — Collect the Full Streamed Answer
import asyncio

async def stream_and_collect(question: str) -> str:
    answer_tokens = []
    current_node  = None

    async for event in app2.astream_events(
        {"question": question, "thinking": "", "answer": ""},
        version="v1",
    ):
        if event["event"] == "on_node_start":
            current_node = event["name"]
        elif event["event"] == "on_chat_model_stream":
            chunk = event["data"]["chunk"].content
            if chunk and current_node == "answer_node":
                answer_tokens.append(chunk)
                print(chunk, end="", flush=True)

    print()
    return "".join(answer_tokens)

full_answer = await stream_and_collect("What is LangGraph?")
print(f"\nCollected answer ({len(full_answer)} chars): {full_answer[:80]}...")

In [ ]:
# Answer 3 — Progress Indicator
import asyncio

async def stream_with_progress(question: str):
    async for event in app2.astream_events(
        {"question": question, "thinking": "", "answer": ""},
        version="v1",
    ):
        if event["event"] == "on_node_start":
            node = event["name"]
            if node == "think_node":
                print("\n[thinking...]", end=" ", flush=True)
            elif node == "answer_node":
                print("\n[answering...]", end=" ", flush=True)
        elif event["event"] == "on_chat_model_stream":
            chunk = event["data"]["chunk"].content
            if chunk:
                print(chunk, end="", flush=True)
    print()

await stream_with_progress("Explain gradient descent in 2 sentences.")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*